# Chapter 7: Finetuning to Follow Instructions

Reference: `LLMs-from-scratch/ch07/`

**Goals:**
- Load an instruction-following dataset (Alpaca-style)
- Format prompts with instruction templates
- Finetune the model for instruction following
- Evaluate with qualitative examples
- (Bonus) Implement DPO for preference alignment

In [15]:
import sys; sys.path.append('..')
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import tiktoken
import json
import os
from functools import partial
from tqdm import tqdm
import numpy as np
from src.models import GPTModel, generate_text_simple
from src.config import GPT_CONFIG_124M
from src.gpt_download import download_and_load_gpt2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 1. Load Instruction Dataset

In [16]:
# Download instruction dataset
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch07/01_main-chapter-code/instruction-data.json"
file_path = "../data/instruction-data.json"

if not os.path.exists(file_path):
    import urllib.request
    urllib.request.urlretrieve(url, file_path)
    print("Downloaded instruction dataset.")
else:
    print("Dataset already exists.")

with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Total examples: {len(data)}")
print(f"First example:\n{json.dumps(data[0], indent=2)}")

# Split
train_portion = int(len(data) * 0.85)
test_portion = int(len(data) * 0.1)
train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]

print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

Dataset already exists.
Total examples: 1100
First example:
{
  "instruction": "Evaluate the following phrase by transforming it into the spelling given.",
  "input": "freind --> friend",
  "output": "The spelling of the given phrase \"freind\" is incorrect, the correct spelling is \"friend\"."
}
Train: 935, Val: 55, Test: 110


## 2. Prompt Formatting 

In [17]:
def format_input(entry):
    instruction = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    return instruction + input_text

# Test formatting
print("Formatted example:\n")
print(format_input(data[0]) + f"\n\n### Response:\n{data[0]['output']}")

Formatted example:

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Evaluate the following phrase by transforming it into the spelling given.

### Input:
freind --> friend

### Response:
The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".


## 3. Instruction Dataset + Loss Masking 

In [18]:
# InstructionDataset class + custom_collate_fn with prompt masking
class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.encoded_texts = []
        self.instruction_lengths = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            
            self.encoded_texts.append(tokenizer.encode(full_text))
            self.instruction_lengths.append(len(tokenizer.encode(instruction_plus_input)))

    def __getitem__(self, idx):
        return self.instruction_lengths[idx], self.encoded_texts[idx]

    def __len__(self):
        return len(self.encoded_texts)

def custom_collate_fn(batch, pad_token_id=50256, ignore_index=-100, allowed_max_length=1024, device="cpu"):
    # Find the longest sequence in the batch (+1 for the <|endoftext|> we append)
    batch_max_length = max(len(item) + 1 for _, item in batch)

    inputs_lst, targets_lst = [], []

    for instruction_length, item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]  # append <|endoftext|> as end-of-response marker
        
        # Pad to batch_max_length
        padded = new_item + [pad_token_id] * (batch_max_length - len(new_item))
        inputs = torch.tensor(padded[:-1])   # everything except last token
        targets = torch.tensor(padded[1:])   # shifted right by 1

        # Mask padding tokens in targets with ignore_index
        # Keep the FIRST pad token (it's the <|endoftext|> the model should learn to predict)
        # but mask all subsequent pad tokens
        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        # Mask all input and instruction tokens in the targets
        # Since targets are shifted by 1, prompt tokens end at instruction_length - 1
        targets[:instruction_length - 1] = ignore_index

        # Truncate to max allowed length
        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    return inputs_tensor, targets_tensor

In [19]:
# Weight loading helpers (same as notebook 05)
def assign(left, right):
    if left.shape != right.shape:
        raise ValueError(f"Shape mismatch. Left: {left.shape}, Right: {right.shape}")
    return torch.nn.Parameter(torch.tensor(right))

def load_weights_into_gpt(gpt, params):
    gpt.pos_emb.weight = assign(gpt.pos_emb.weight, params["wpe"])
    gpt.tok_emb.weight = assign(gpt.tok_emb.weight, params["wte"])

    for b in range(len(params["blocks"])):
        q_w, k_w, v_w = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["w"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.weight = assign(
            gpt.trf_blocks[b].att.W_query.weight, q_w.T)
        gpt.trf_blocks[b].att.W_key.weight = assign(
            gpt.trf_blocks[b].att.W_key.weight, k_w.T)
        gpt.trf_blocks[b].att.W_value.weight = assign(
            gpt.trf_blocks[b].att.W_value.weight, v_w.T)

        q_b, k_b, v_b = np.split(
            (params["blocks"][b]["attn"]["c_attn"])["b"], 3, axis=-1)
        gpt.trf_blocks[b].att.W_query.bias = assign(
            gpt.trf_blocks[b].att.W_query.bias, q_b)
        gpt.trf_blocks[b].att.W_key.bias = assign(
            gpt.trf_blocks[b].att.W_key.bias, k_b)
        gpt.trf_blocks[b].att.W_value.bias = assign(
            gpt.trf_blocks[b].att.W_value.bias, v_b)

        gpt.trf_blocks[b].att.out_proj.weight = assign(
            gpt.trf_blocks[b].att.out_proj.weight,
            params["blocks"][b]["attn"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].att.out_proj.bias = assign(
            gpt.trf_blocks[b].att.out_proj.bias,
            params["blocks"][b]["attn"]["c_proj"]["b"])

        gpt.trf_blocks[b].ff.layers[0].weight = assign(
            gpt.trf_blocks[b].ff.layers[0].weight,
            params["blocks"][b]["mlp"]["c_fc"]["w"].T)
        gpt.trf_blocks[b].ff.layers[0].bias = assign(
            gpt.trf_blocks[b].ff.layers[0].bias,
            params["blocks"][b]["mlp"]["c_fc"]["b"])
        gpt.trf_blocks[b].ff.layers[2].weight = assign(
            gpt.trf_blocks[b].ff.layers[2].weight,
            params["blocks"][b]["mlp"]["c_proj"]["w"].T)
        gpt.trf_blocks[b].ff.layers[2].bias = assign(
            gpt.trf_blocks[b].ff.layers[2].bias,
            params["blocks"][b]["mlp"]["c_proj"]["b"])

        gpt.trf_blocks[b].norm1.scale = assign(
            gpt.trf_blocks[b].norm1.scale, params["blocks"][b]["ln_1"]["g"])
        gpt.trf_blocks[b].norm1.shift = assign(
            gpt.trf_blocks[b].norm1.shift, params["blocks"][b]["ln_1"]["b"])
        gpt.trf_blocks[b].norm2.scale = assign(
            gpt.trf_blocks[b].norm2.scale, params["blocks"][b]["ln_2"]["g"])
        gpt.trf_blocks[b].norm2.shift = assign(
            gpt.trf_blocks[b].norm2.shift, params["blocks"][b]["ln_2"]["b"])

    gpt.final_norm.scale = assign(gpt.final_norm.scale, params["g"])
    gpt.final_norm.shift = assign(gpt.final_norm.shift, params["b"])
    gpt.out_head.weight = assign(gpt.out_head.weight, params["wte"])

# Build dataloaders
tokenizer = tiktoken.get_encoding("gpt2")

train_dataset = InstructionDataset(train_data, tokenizer)
val_dataset = InstructionDataset(val_data, tokenizer)

customized_collate_fn = partial(custom_collate_fn, device=device, allowed_max_length=1024)

torch.manual_seed(123)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True,
    collate_fn=customized_collate_fn, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False,
    collate_fn=customized_collate_fn, drop_last=False)

# Load GPT-2 124M weights
settings, params = download_and_load_gpt2("124M", models_dir="../gpt2")
model = GPTModel(GPT_CONFIG_124M)
load_weights_into_gpt(model, params)
model.to(device)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")
print("Model loaded successfully.")

# Sanity check: peek at one batch
for inputs, targets in train_loader:
    print(f"Input shape: {inputs.shape}, Target shape: {targets.shape}")
    print(f"Masked targets (count of -100): {(targets == -100).sum().item()}")
    break

File already exists and is up-to-date: ../gpt2\124M\checkpoint
File already exists and is up-to-date: ../gpt2\124M\encoder.json
File already exists and is up-to-date: ../gpt2\124M\hparams.json
File already exists and is up-to-date: ../gpt2\124M\model.ckpt.data-00000-of-00001
File already exists and is up-to-date: ../gpt2\124M\model.ckpt.index
File already exists and is up-to-date: ../gpt2\124M\model.ckpt.meta
File already exists and is up-to-date: ../gpt2\124M\vocab.bpe
Train batches: 116, Val batches: 7
Model loaded successfully.
Input shape: torch.Size([8, 62]), Target shape: torch.Size([8, 62])
Masked targets (count of -100): 354


In [20]:
# Generate function with temperature, top-k, and EOS stopping
def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(logits < min_val, torch.tensor(float("-inf")).to(logits.device), logits)

        if temperature > 0.0:
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        if eos_id is not None and idx_next == eos_id:
            break

        idx = torch.cat((idx, idx_next), dim=1)
    return idx

# Loss helpers (same as your src/train.py but these work with the collate_fn's ignore_index=-100)
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    # cross_entropy ignores targets == -100 by default, so loss masking works automatically
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    return loss

def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

In [21]:
# Instruction Finetuning Loop
def train_instruction_model(model, train_loader, val_loader, optimizer, device,
                            num_epochs, eval_freq, eval_iter, tokenizer, sample_prompt):
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()
            tokens_seen += input_batch.numel()
            global_step += 1

            if global_step % eval_freq == 0:
                model.eval()
                with torch.no_grad():
                    train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
                    val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch+1} (Step {global_step:06d}): "
                      f"Train loss {train_loss:.3f}, Val loss {val_loss:.3f}")

        # Generate a sample response after each epoch to check progress
        model.eval()
        with torch.no_grad():
            sample_ids = tokenizer.encode(sample_prompt)
            sample_tensor = torch.tensor(sample_ids, device=device).unsqueeze(0)
            output_ids = generate(model, sample_tensor, max_new_tokens=256,
                                  context_size=GPT_CONFIG_124M["context_length"],
                                  temperature=0.0, eos_id=50256)
            generated_text = tokenizer.decode(output_ids[0].tolist())
            print(f"\n--- Epoch {epoch+1} sample response ---")
            print(generated_text)
            print("---\n")

    return train_losses, val_losses, track_tokens_seen

# Print initial losses
model.eval()
with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device, num_batches=5)
print(f"Initial train loss: {train_loss:.3f}")
print(f"Initial val loss:   {val_loss:.3f}")

# Train the model (2 epochs SFT)
import time
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)

# Use the first validation example as a sample prompt during training
sample_prompt = format_input(val_data[0]) + "\n\n### Response:\n"

torch.manual_seed(123)
start_time = time.time()

train_losses, val_losses, tokens_seen = train_instruction_model(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=2, eval_freq=5, eval_iter=5,
    tokenizer=tokenizer, sample_prompt=sample_prompt
)

end_time = time.time()
print(f"\nTraining completed in {(end_time - start_time) / 60:.2f} minutes.")

Initial train loss: 3.007
Initial val loss:   2.811
Ep 1 (Step 000000): Train loss 1.854, Val loss 1.805
Ep 1 (Step 000005): Train loss 1.324, Val loss 1.219
Ep 1 (Step 000010): Train loss 1.094, Val loss 1.114
Ep 1 (Step 000015): Train loss 1.106, Val loss 1.083
Ep 1 (Step 000020): Train loss 1.038, Val loss 1.050
Ep 1 (Step 000025): Train loss 0.995, Val loss 1.038
Ep 1 (Step 000030): Train loss 0.954, Val loss 1.017
Ep 1 (Step 000035): Train loss 0.820, Val loss 0.996
Ep 1 (Step 000040): Train loss 1.005, Val loss 0.979
Ep 1 (Step 000045): Train loss 0.726, Val loss 0.976
Ep 1 (Step 000050): Train loss 0.882, Val loss 0.964
Ep 1 (Step 000055): Train loss 1.111, Val loss 0.953
Ep 1 (Step 000060): Train loss 0.854, Val loss 0.949
Ep 1 (Step 000065): Train loss 0.761, Val loss 0.945
Ep 1 (Step 000070): Train loss 0.740, Val loss 0.940
Ep 1 (Step 000075): Train loss 0.704, Val loss 0.930
Ep 1 (Step 000080): Train loss 0.728, Val loss 0.919
Ep 1 (Step 000085): Train loss 0.717, Val loss 

## 4. Generate & Evaluate Responses

In [22]:
# Test the finetuned model on a few examples
model.eval()
print("=" * 60)
print("INSTRUCTION-FINETUNED MODEL RESPONSES")
print("=" * 60)

for i, entry in enumerate(test_data[:5]):
    input_text = format_input(entry)
    prompt = input_text + "\n\n### Response:\n"

    token_ids = tokenizer.encode(prompt)
    token_tensor = torch.tensor(token_ids, device=device).unsqueeze(0)

    with torch.no_grad():
        output_ids = generate(
            model, token_tensor,
            max_new_tokens=256,
            context_size=GPT_CONFIG_124M["context_length"],
            temperature=0.0,
            eos_id=50256
        )
    generated_text = tokenizer.decode(output_ids[0].tolist())
    response = generated_text[len(prompt):].strip()

    print(f"\n{'─' * 50}")
    print(f"Instruction: {entry['instruction'][:80]}...")
    if entry["input"]:
        print(f"Input: {entry['input'][:60]}...")
    print(f"\nExpected: {entry['output'][:100]}...")
    print(f"Model:    {response[:100]}...")

INSTRUCTION-FINETUNED MODEL RESPONSES

──────────────────────────────────────────────────
Instruction: Rewrite the sentence using a simile....
Input: The car is very fast....

Expected: The car is as fast as lightning....
Model:    The car is as fast as a cheetah....

──────────────────────────────────────────────────
Instruction: What type of cloud is typically associated with thunderstorms?...

Expected: The type of cloud typically associated with thunderstorms is cumulonimbus....
Model:    The type of cloud typically associated with thunderstorms is a cumulus cloud....

──────────────────────────────────────────────────
Instruction: Name the author of 'Pride and Prejudice'....

Expected: Jane Austen....
Model:    The author of 'Pride and Prejudice' is William Shakespeare....

──────────────────────────────────────────────────
Instruction: What is the periodic symbol for chlorine?...

Expected: The periodic symbol for chlorine is Cl....
Model:    The periodic symbol for chlorine is C

## 5. Save Model

In [23]:
# Save the instruction-finetuned model
save_path = "../checkpoints/model_instruction_finetuned.pth"
torch.save(model.state_dict(), save_path)
print(f"Model saved to {save_path}")

# Also save the full responses for later review
responses = []
model.eval()
for entry in tqdm(test_data, desc="Generating responses"):
    input_text = format_input(entry)
    prompt = input_text + "\n\n### Response:\n"
    token_ids = tokenizer.encode(prompt)
    token_tensor = torch.tensor(token_ids, device=device).unsqueeze(0)

    with torch.no_grad():
        output_ids = generate(
            model, token_tensor, max_new_tokens=256,
            context_size=GPT_CONFIG_124M["context_length"],
            temperature=0.0, eos_id=50256
        )
    generated_text = tokenizer.decode(output_ids[0].tolist())
    response = generated_text[len(prompt):].strip()
    responses.append({
        "instruction": entry["instruction"],
        "input": entry["input"],
        "expected": entry["output"],
        "model_response": response
    })

with open("../data/instruction-responses.json", "w", encoding="utf-8") as f:
    json.dump(responses, f, indent=2)
print(f"Saved {len(responses)} responses to ../data/instruction-responses.json")

Model saved to ../checkpoints/model_instruction_finetuned.pth


Generating responses: 100%|██████████| 110/110 [00:30<00:00,  3.57it/s]

Saved 110 responses to ../data/instruction-responses.json
